In [0]:
from pyspark.sql.session import SparkSession

def get_spark_session(app_name="sample data engineering project"):
    try:
        spark=SparkSession.getActiveSession()
        if spark:
            return spark
    except:
        pass
    return (SparkSession.builder.config("spark.sql.shuffle.partitions", "1").appName(app_name).getOrCreate())

###BUSINESS FUNCTION

In [0]:
from pyspark.sql import functions as F
# CLEANSING

def dropnull(df,subset=None,how="any"):
    return df.na.drop(subset=subset,how=how)

# ADDING SOURCE FILE NAME

def add_source(df,srcname):
    return df.withColumn("data_source",F.lit(srcname))

# REPLACE VALUES
def replace_values(df,val,subset=None):
    return df.na.replace(val,subset=subset)

# STD

def shipment_std(df):
    return (df.withColumn("domain",F.lit("Logistics"))
            .withColumn("ingestion_timestamp",F.current_timestamp())
            .withColumn("is_expedited",False)
            .withColumn("shipment_date",F.to_date(F.col("shipment_date"),"yy-MM-dd"))
            .withColumn("shipment_cost",F.round(F.col("shipment_cost"),2))
            .withColumn("shipment_weight_kg",F.col("shipment_weight_kg").cast("double")))


def staff_std(df):
    return(df
           .withColumn("role",F.lower("role"))
           .withColumn("vehicle_type",F.upper(F.col("vehicle_type")))
           .withColumn("age",F.col("age").cast("int"))
           .withColumn("load_dt",F.current_timestamp())
           .withColumn("full_name",F.concat_ws(" ",col("first_name"),col("last_name")))
           .drop("first_name","last_name")
           .withColumnRenamed("full_name","staff_full_name"))
    


def enrich_shipment(df):
    return(df
           .withColumn("route_segment",F.concat_ws("-",col("source_city"),col("destination_city")))
           .withColumn("vehicle_identifier",F.concat_ws("_",col("vehicle_type"),col("shipment_id")))
           .withColumn("shipment_year",F.year(F.col("shipment_date")))
           .withColumn("shipment_month",F.month(F.col("shipment_date")))
           .withColumn("is_weekend",F.dayofweek(F.col("shipment_date").isin([1,7])))
           )




In [0]:
# UNION

def union(df1,df2):
    return df1.unionByName(df2,allowMissingColumns=True)

###COMMON FUNCTION

In [0]:
# READ FILE:
def read_file(spark,path,sep=",",header=True,fmt="csv"):
    df=spark.read.options(sep=sep,header=header,inferSchema=True).format(fmt).load(path)
    return df

# READ JSON:
def read_json(spark,path):
    df=spark.read.json(path,multiLine=True,mode="permissive")
    return df

# DELTA FILE READ

def read_delta(spark,path):
    df=spark.read.format("delta").load(path)
    return df

# READ TABLE
def read_table(spark,table):
    df=spark.table(table)
    return df


In [0]:
# WRITE FILE
def write_file(df,path):
    return df.spark.write.mode("overwrite").format("delta").save(path)

# WRITE TABLE

def write_tanle(df,path):
    return df.spark.mode("overwrite").format("delta").saveAsTable(path)